# Why entanglement cannot signal faster than light

Entanglement can produce correlations between distant measurements that no classical local model can reproduce. It does **not**, however, give either observer control over the other observer's local measurement statistics. This distinction is the content of the **no-signalling theorem**.

The Heisenberg picture makes the distinction especially direct. We keep the reference state fixed and evolve the observables. Once Alice and Bob's qubits have stopped interacting, an operation performed on Alice's qubit can change Alice's observables and the correlations represented by joint observables, but it cannot change any observable that Bob can measure locally.

This notebook develops that statement using a Bell pair, then extends it from unitary gates to arbitrary local quantum operations.

## Prepare the correlations while the qubits are together

Alice and Bob begin with two qubits in the fixed reference state $\lvert 00\rangle$. Before separating them, they apply a Hadamard gate to Alice's qubit followed by a CNOT from Alice to Bob. In the Schrödinger picture this prepares

$$\lvert\Phi^+\rangle = \frac{\lvert 00\rangle+\lvert 11\rangle}{\sqrt{2}}.$$

In the Heisenberg picture the state stays $\lvert 00\rangle$. The same gates instead transform the observables. Lower-case symbols such as $X_a$ and $Z_b$ below denote the time-zero Pauli observables in that fixed reference state.

<img src="images/bell-state-circuit.svg" alt="Bell-pair preparation: Hadamard on Alice followed by CNOT from Alice to Bob, then a later measurement of each qubit" width="360"/>

In [ ]:
from IPython.display import display
from sympy import Rational

from gates_multi import cnot
from gates_single import hadamard
from measurement import expectation_in_zero_state
from qubit import Qubit
from sympy_pauli import evaluate_labelled_pauli_product

alice_0 = Qubit.qubit_time_0("a")
bob_0 = Qubit.qubit_time_0("b")

alice_bell, bob_bell = cnot(hadamard(alice_0), bob_0)

display(alice_bell)
display(bob_bell)

The output says that the relevant observables have evolved to

$$X_A(t)=Z_aX_b, \qquad Z_A(t)=X_a,$$

$$X_B(t)=X_b, \qquad Z_B(t)=X_aZ_b.$$

No individual observable is sharp in the fixed state $\lvert 00\rangle$: each local X or Z measurement has expectation value zero and therefore gives $+1$ or $-1$ with equal probability. The products are different:

$$X_A(t)X_B(t)=Z_a, \qquad Z_A(t)Z_B(t)=Z_b.$$

Both $Z_a$ and $Z_b$ have expectation value $+1$ in $\lvert 00\rangle$. Alice's and Bob's results are therefore perfectly correlated when they both measure X or both measure Z, even though neither observer can predict their own result.

In [ ]:
labels = ["a", "b"]


def expectation(expr):
    simplified = evaluate_labelled_pauli_product(expr)
    return expectation_in_zero_state(simplified, labels)


local_expectations = {
    "<X_A>": expectation(alice_bell.x),
    "<Z_A>": expectation(alice_bell.z),
    "<X_B>": expectation(bob_bell.x),
    "<Z_B>": expectation(bob_bell.z),
}

joint_expectations = {
    "<X_A X_B>": expectation(alice_bell.x * bob_bell.x),
    "<Z_A Z_B>": expectation(alice_bell.z * bob_bell.z),
}

bob_probabilities = {
    "P(X_B = +1)": expectation(Rational(1, 2) * (1 + bob_bell.x)),
    "P(X_B = -1)": expectation(Rational(1, 2) * (1 - bob_bell.x)),
    "P(Z_B = +1)": expectation(Rational(1, 2) * (1 + bob_bell.z)),
    "P(Z_B = -1)": expectation(Rational(1, 2) * (1 - bob_bell.z)),
}

display(local_expectations)
display(joint_expectations)
display(bob_probabilities)

## Separate the qubits, then let Alice choose an operation

Suppose Alice and Bob now carry their qubits far apart. Alice would like to send a bit by either doing nothing or applying a unitary $U_A$. Could Bob determine her choice from measurements on his qubit?

For any Bob-local observable $O_B$, the corresponding operator on the joint system is $I_A\otimes O_B$. In the Heisenberg picture Alice's operation transforms it as

$$
(U_A^\dagger\otimes I_B)(I_A\otimes O_B)(U_A\otimes I_B)
= (U_A^\dagger U_A)\otimes O_B
= I_A\otimes O_B.
$$

Alice's operation commutes with Bob's observable because the two operators act on different subsystems. Consequently, Alice cannot change the expectation value or probability distribution of **any** measurement available to Bob—not merely measurements in the computational basis.

The compound expressions created earlier do not contradict this. They record the common interaction that prepared the Bell pair. After the qubits separate, a local gate evolves Alice's descriptor but does not rewrite Bob's descriptor.

In [ ]:
alice_after_local_h = hadamard(alice_bell)

print("Alice's descriptor after her local Hadamard:")
display(alice_after_local_h)

print("Bob's unchanged descriptor:")
display(bob_bell)

bob_probabilities_after_alice_h = {
    "P(X_B = +1)": expectation(Rational(1, 2) * (1 + bob_bell.x)),
    "P(X_B = -1)": expectation(Rational(1, 2) * (1 - bob_bell.x)),
    "P(Z_B = +1)": expectation(Rational(1, 2) * (1 + bob_bell.z)),
    "P(Z_B = -1)": expectation(Rational(1, 2) * (1 - bob_bell.z)),
}

assert bob_probabilities_after_alice_h == bob_probabilities
display(bob_probabilities_after_alice_h)

## Measurement changes conditional predictions, not Bob's local statistics

The apparent paradox usually returns when Alice measures her qubit. Let $A_r$ be Alice's projector for outcome $r$ and $B_s$ Bob's projector for outcome $s$. In the Heisenberg picture their joint probability is

$$p(r,s)=\langle 00\rvert A_r(t)B_s(t)\lvert 00\rangle.$$

If somebody later tells us that Alice obtained $r$, we can form the conditional probability

$$p(s\mid r)=\frac{p(r,s)}{p(r)}.$$

For the Bell pair, Alice's result can make this conditional probability certain: if both observers measure Z, Bob's result equals Alice's. But Bob does not initially know $r$, and Alice cannot choose it. From Bob's local point of view Alice's alternatives must be summed:

$$
\begin{aligned}
p_B(s)
&=\sum_r p(r,s)\\
&=\left\langle 00\middle\rvert\left(\sum_r A_r(t)\right)B_s(t)\middle\lvert 00\right\rangle\\
&=\langle 00\rvert B_s(t)\lvert 00\rangle,
\end{aligned}
$$

because a complete measurement obeys $\sum_r A_r(t)=I$. Alice's measurement therefore changes the correlations and our predictions **conditioned on her record**, but it cannot alter the unconditioned frequencies in Bob's log.

Only after Alice's result reaches Bob through an ordinary classical channel can they sort their records into matching branches and expose the correlations.

## The result is not restricted to unitary gates

A general local operation performed by Alice is a trace-preserving quantum channel $\mathcal E_A$. In the Heisenberg picture observables evolve under its dual channel $\mathcal E_A^\dagger$. Trace preservation makes the dual channel **unital**:

$$\mathcal E_A^\dagger(I_A)=I_A.$$

It follows that every Bob-local observable is still fixed:

$$
(\mathcal E_A^\dagger\otimes\mathrm{id}_B)(I_A\otimes O_B)
=\mathcal E_A^\dagger(I_A)\otimes O_B
=I_A\otimes O_B.
$$

This includes measurements when Alice's outcome is ignored. If Alice postselects a particular outcome, Bob's conditional ensemble may differ, but Bob cannot know which trials Alice retained until she sends the success flag. Postselection therefore does not provide a hidden faster-than-light channel.

## Decoherence spreads the record into the environment

A laboratory measurement does not leave its result inside one microscopic apparatus qubit. The apparatus interacts locally with air molecules, thermal radiation, detector electronics and many other degrees of freedom. Those systems then interact with still more systems. In this sense the measurement record “infects” billions of environmental atoms and photons.

The word “infects” is useful imagery, but the physical process is ordinary local entangling interaction. A simple model treats the measured qubit as the control of a succession of CNOTs whose targets begin in $\lvert 0\rangle$. If the measured alternatives are the orthogonal pointer states $\lvert 0\rangle$ and $\lvert 1\rangle$, each interaction imprints the same pointer value on another environmental degree of freedom:

$$
\alpha\lvert 0\rangle\lvert 0\cdots 0\rangle_E
+\beta\lvert 1\rangle\lvert 1\cdots 1\rangle_E.
$$

This does **not** clone an arbitrary quantum state. It redundantly records one effectively classical, orthogonal pointer observable. Phase information remains encoded in global compound observables and rapidly becomes inaccessible to any observer who controls only a tiny fraction of the environment. That loss of locally accessible phase coherence is decoherence.

In the Heisenberg picture the same process appears as the growth of compound observables. Each environmental pointer's Z observable acquires the measured qubit's Z observable as a factor. Further local interactions add more carriers of the same record while the fixed reference state remains unchanged.

In [ ]:
alice_environment = [Qubit.qubit_time_0(f"eA{i}") for i in range(1, 4)]
bob_environment = [Qubit.qubit_time_0(f"eB{i}") for i in range(1, 4)]

alice_with_records = alice_bell
alice_records = []
for environment_qubit in alice_environment:
    alice_with_records, record = cnot(alice_with_records, environment_qubit)
    alice_records.append(record)

bob_with_records = bob_bell
bob_records = []
for environment_qubit in bob_environment:
    bob_with_records, record = cnot(bob_with_records, environment_qubit)
    bob_records.append(record)

print("Three fragments of Alice's local environment now carry her Z record:")
for record in alice_records:
    display(record.z)

print("Three fragments of Bob's local environment now carry his Z record:")
for record in bob_records:
    display(record.z)

The example uses only three environmental qubits on each side, but repeating the same local interaction gives an arbitrarily large record. In the output, every Alice-side record contains the factor $X_a=Z_A(t)$, while every Bob-side record contains $X_aZ_b=Z_B(t)$. Consequently:

- Any one local fragment can act as a durable record of its nearby measurement outcome.
- Different fragments on the same side agree about that outcome.
- An Alice-side fragment and a Bob-side fragment retain the original Bell-pair correlation.
- No individual fragment reveals the distant result or allows either observer to select their own random result.

The correlation is therefore not a fragile coincidence that disappears when a microscopic qubit meets a macroscopic detector. It is redistributed into correlations among a vast number of degrees of freedom. Destroying or losing a few photons does not erase the fact because many other fragments carry redundant records. This redundancy is closely related to **environment-induced superselection** and **quantum Darwinism**: stable pointer facts become widely available because the environment contains many mutually consistent copies of the record.

All of this spreading occurs through local interactions at or below the speed of light. Alice's environment becomes correlated with Alice's record, and Bob's environment becomes correlated with Bob's record. The cross-laboratory relationship becomes operationally visible only when some physical signal brings records from the two sides together. Decoherence makes the records robust and effectively classical; it does not turn their correlation into a controllable communication channel.

## What actually travels?

The entangling interaction happens while the qubits are together and establishes compound observables that encode future correlations. After separation:

- Alice's local choices evolve Alice-local observables.
- Bob's local observables and all of Bob's unconditioned measurement probabilities remain unchanged.
- Joint observables can reveal non-classical correlations when the two records are brought together.
- The information needed to identify those correlations—Alice's choice, basis and outcome—must travel through a classical channel, no faster than light.

Nothing in this argument requires a distant particle to undergo a locally detectable instantaneous change. A state-update or “collapse” rule can be used to calculate conditional predictions, but it does not represent a controllable signal arriving at Bob. In the Heisenberg representation the operational reason is visible in the algebra: after the qubits cease interacting, Alice cannot evolve an observable that Bob can measure locally.